# Model architecture

## Imports

In [1]:
from pathlib import Path
import sys
import torch
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "project2" / "src" / "models.py").exists()
)

sys.path.insert(0, str(root / "project2" / "src"))

from models import ARTPairwiseBERT, CHECKPOINT
from data_loader import ARTDataset, ARTCollator

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
print("Checkpoint:", CHECKPOINT)

C:\Users\kimbo\anaconda3\envs\nlp-art-hs26\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Checkpoint: google-bert/bert-base-uncased


### Compare both architectures

In [2]:
# Create both variants on CPU to conserve GPU memory.
random_model = ARTPairwiseBERT("random")
pretrained_model = ARTPairwiseBERT("pretrained")

random_state = random_model.state_dict()
pretrained_state = pretrained_model.state_dict()

# Same parameter names and tensor shapes
assert random_state.keys() == pretrained_state.keys()

for name in random_state:
    assert random_state[name].shape == pretrained_state[name].shape

# Same parameter count
random_params = sum(p.numel() for p in random_model.parameters())
pretrained_params = sum(p.numel() for p in pretrained_model.parameters())

assert random_params == pretrained_params

# Verify that encoder initializations actually differ.
embedding_key = "encoder.embeddings.word_embeddings.weight"

assert not torch.equal(
    random_state[embedding_key],
    pretrained_state[embedding_key]
)

print("Random parameters:", f"{random_params:,}")
print("Pretrained parameters:", f"{pretrained_params:,}")
print("Architecture match: PASS")
print("Different encoder initialization: PASS")

C:\Users\kimbo\anaconda3\envs\nlp-art-hs26\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kimbo\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\kimbo\anaconda3\envs\nlp-art-hs26\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache

Random parameters: 108,892,417
Pretrained parameters: 108,892,417
Architecture match: PASS
Different encoder initialization: PASS


### Verify identical classifier initialization

In [3]:
def reset_classifier(model, seed=1234):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        model.classifier.reset_parameters()

reset_classifier(random_model)
reset_classifier(pretrained_model)

assert torch.equal(
    random_model.classifier.weight,
    pretrained_model.classifier.weight
)

assert torch.equal(
    random_model.classifier.bias,
    pretrained_model.classifier.bias
)

print("Classifier initialization match: PASS")

Classifier initialization match: PASS


### Pretrained BERT forward-pass test

In [4]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

dataset = ARTDataset(
    "train",
    tokenizer,
    max_length=128
)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=ARTCollator(tokenizer.pad_token_id)
)

batch = next(iter(loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

# Only move the pretrained model to GPU.
pretrained_model = pretrained_model.to(device)
pretrained_model.eval()

with torch.inference_mode():
    logits = pretrained_model(input_ids, attention_mask)

    swapped_logits = pretrained_model(
        input_ids.flip(dims=[1]),
        attention_mask.flip(dims=[1])
    )

assert logits.shape == (2, 2)

assert torch.allclose(
    swapped_logits,
    logits.flip(dims=[1]),
    atol=1e-5,
    rtol=1e-4
)

print("Input shape:", tuple(input_ids.shape))
print("Output shape:", tuple(logits.shape))
print("Pretrained forward pass: PASS")
print("Hypothesis swap: PASS")

Input shape: (2, 2, 37)
Output shape: (2, 2)
Pretrained forward pass: PASS
Hypothesis swap: PASS
